# Data engineering and audio exploration
Real PASCAL + PhysioNet labels. Artifact, unlabeled, poor-SQI and duplicate recordings are excluded with recorded reasons. Groups protect acquisition/recording boundaries, not verified patient identities. Only the first five seconds enter this versioned benchmark; do not claim whole-recording sensitivity.

In [ ]:
from pathlib import Path
import sys
if not Path("training").exists():
    sys.path.insert(0, str(Path.cwd().parent))
    import os
    os.chdir(Path.cwd().parent)
import pandas as pd
from training.data import write_manifest
records = write_manifest(Path("datasets"), Path("vault/manifests/audio.json"))
df = pd.DataFrame(records)
display(df.groupby(["source", "split", "label"]).size().unstack(fill_value=0))
assert df.groupby("group")["split"].nunique().max() == 1
assert not df.sha256.duplicated().any()

In [ ]:
import json
manifest = json.loads(Path("vault/manifests/audio.json").read_text())
display(pd.DataFrame(manifest["excluded"]).groupby("reason").size())
df.groupby(["source", "label"]).size().unstack().plot.bar(title="Source/class distribution")

In [ ]:
import matplotlib.pyplot as plt
from audio_engine.contract import decode, features, RATE
content = Path(df.iloc[0].path).read_bytes()
signal = decode(content)
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot([i/RATE for i in range(len(signal))], signal)
axes[0].set(xlabel="Seconds", ylabel="Normalized amplitude")
axes[1].imshow(features(content, "mel")[..., 0], origin="lower", aspect="auto")
axes[1].set(xlabel="Time frame", ylabel="Mel band")
plt.tight_layout()